<div align="center">

# NB7 · From Explanation to Governance

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · Grand Forks, ND, USA · 28 October 2026 · Session 11:30 to 11:50

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB7_Governance_and_Dashboard.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

The earlier notebooks produced explanations, evaluations of explanations and evidence of attacks. This notebook turns them into the artefacts that regulators, hospital committees and auditors ask for: a map of where a clinical system sits under the EU AI Act, a subgroup audit with uncertainty, a model card, and a dashboard that puts the explanation in front of the clinician. The running example is the heart disease model of NB2, trained on the same UCI data [10]. The notebook runs in about a minute.

In [ ]:
import importlib.util, subprocess, sys
PACKAGES = {"shap": "shap>=0.46", "tabulate": "tabulate"}
missing = [spec for module, spec in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import shap
import ipywidgets as widgets
from IPython.display import display, Markdown
from matplotlib.patches import Polygon
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

## 1. Where a clinical AI system sits

The EU AI Act sorts AI systems into four levels of risk [1]. Software that is a medical device, or a safety component of one, and that requires a notified body under the Medical Device Regulation [3] is high-risk through Annex I. The Digital Omnibus on AI [2] moved the start of the high-risk obligations, and the dates below are those in force when the material was checked on 28 September 2026.

In [ ]:
tiers = [("Minimal risk", "most other AI, e.g. scheduling or spam filtering;\nvoluntary codes of conduct", SAFE),
         ("Transparency", "chatbots and generated content must be disclosed\n(Article 50, applies from 2 August 2026)", VIOLET),
         ("High risk", "AI in medical devices (Annex I, from 2 August 2028); emergency triage\nand access to essential health services (Annex III, from 2 December 2027)", AMBER),
         ("Unacceptable", "prohibited practices such as social scoring\n(banned since 2 February 2025)", ALERT)]
fig, ax = plt.subplots(figsize=(12, 4.8))
for i, (name, text, col) in enumerate(tiers):
    w0, w1 = 4 - i, 4 - (i + 1) if i < 3 else 0.0
    ax.add_patch(Polygon([(-w0, i), (w0, i), (w1, i + 1), (-w1, i + 1)], closed=True, facecolor=col,
                         alpha=0.85, edgecolor="white", lw=2))
    ax.text(0, i + 0.5, name, ha="center", va="center", color="white", fontsize=11, fontweight="bold")
    ax.text(4.4, i + 0.5, text, ha="left", va="center", fontsize=9)
ax.annotate("most clinical decision\nsupport systems", xy=(-1.9, 2.5), xytext=(-6.6, 2.9), fontsize=9,
            arrowprops=dict(arrowstyle="->", color=INK))
ax.set_xlim(-7, 12.5)
ax.set_ylim(-0.1, 4.1)
ax.axis("off")
ax.set_title("The EU AI Act risk pyramid and the place of clinical AI")
plt.show()

The obligations for high-risk systems map onto the workshop directly.

| Obligation in the AI Act | What it asks | Where the workshop addressed it |
|---|---|---|
| Article 9, risk management | identify and mitigate foreseeable risks over the whole lifecycle | shortcut learning (NB1), attacks on alerts (NB4 to NB6) |
| Article 10, data governance | relevant, representative data; examination of possible biases | subgroup audit in section 2 of this notebook |
| Article 12, record keeping | automatic logs that allow traceability | monitoring streams and the Page-Hinkley alarm (NB4 to NB6) |
| Article 13, transparency | output that deployers can interpret; instructions for use | local and global explanations (NB2, NB3), model card below |
| Article 14, human oversight | people able to understand, monitor and override | the dashboard in section 4 |
| Article 15, accuracy, robustness, cybersecurity | resilience against attempts to alter use, outputs or performance, including adversarial examples and data poisoning | the three attack scenarios (NB4 to NB6) |

In the United States the corresponding expectations come from the FDA: the joint transparency principles with Health Canada and the MHRA [7], the final guidance on predetermined change control plans [8], and the premarket cybersecurity guidance finalised in February 2026 [9].

## 2. Subgroup audit: does the model work equally well for women and men?

Biases in clinical algorithms have caused real harm, from a care management algorithm that under-served Black patients [5] to chest X-ray models that under-diagnosed women and other groups [6]. The audit below uses out-of-fold predictions for all 297 patients, so that every patient is scored by a model that did not see them, and reports 95 percent bootstrap confidence intervals [12]. With fewer than 100 women in the data, the intervals are wide, and that width is itself a finding.

In [ ]:
UCI_FILE = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
RAW = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
RENAME = {"sex": "sex_male", "cp": "chest_pain_type", "trestbps": "resting_bp", "chol": "cholesterol",
          "fbs": "fasting_glucose_high", "restecg": "resting_ecg", "thalach": "max_heart_rate",
          "exang": "exercise_angina", "oldpeak": "st_depression", "slope": "st_slope",
          "ca": "major_vessels", "thal": "thallium_scan"}


def load_cleveland():
    path = DATA_DIR / "processed.cleveland.data"
    if not path.exists():
        try:
            fetch([UCI_FILE], path)
        except RuntimeError:
            with zipfile.ZipFile(fetch([UCI_ZIP], DATA_DIR / "heart+disease.zip")) as z:
                path.write_bytes(z.read("processed.cleveland.data"))
    return pd.read_csv(path, header=None, names=RAW, na_values="?")


heart = load_cleveland().dropna().rename(columns=RENAME).reset_index(drop=True)
y = (heart.pop("num") > 0).astype(int).to_numpy()
X = heart.astype(float)
FEATURES = list(X.columns)
women = X["sex_male"].to_numpy() == 0


def new_model():
    return GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.9,
                                      random_state=SEED)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(new_model(), X, y, cv=cv, method="predict_proba")[:, 1]
print(f"{len(y)} patients · {women.sum()} women ({y[women].mean():.0%} with disease) · "
      f"{(~women).sum()} men ({y[~women].mean():.0%} with disease)")

In [ ]:
def group_metrics(y_true, p, thr=0.5):
    pred = p >= thr
    tp, fn = (pred & (y_true == 1)).sum(), (~pred & (y_true == 1)).sum()
    fp, tn = (pred & (y_true == 0)).sum(), (~pred & (y_true == 0)).sum()
    return {"AUROC": roc_auc_score(y_true, p), "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1), "positive predictive value": tp / max(tp + fp, 1),
            "observed prevalence": y_true.mean(), "mean predicted risk": p.mean()}


def audit(p, n_boot=200 if QUICK else 1000):
    rs = np.random.default_rng(SEED)
    out = {}
    for name, mask in [("women", women), ("men", ~women)]:
        yt, pt = y[mask], p[mask]
        point = group_metrics(yt, pt)
        boots = []
        for _ in range(n_boot):
            idx = rs.integers(0, len(yt), len(yt))
            if 0 < yt[idx].mean() < 1:
                boots.append(group_metrics(yt[idx], pt[idx]))
        lo, hi = pd.DataFrame(boots).quantile(0.025), pd.DataFrame(boots).quantile(0.975)
        out[f"{name} (n = {mask.sum()})"] = {k: f"{v:.2f} [{lo[k]:.2f}, {hi[k]:.2f}]" for k, v in point.items()}
    return pd.DataFrame(out)


audit_table = audit(p_oof)
audit_table

Differences in sensitivity between groups can reflect the model or simply different prevalence and disease presentation; the overlapping or separated intervals tell which conclusions the data can support. SHAP adds the next layer: which variables drive the predictions in each group, and how much the model uses sex directly.

In [ ]:
model = new_model().fit(X, y)
explainer = shap.TreeExplainer(model, data=X.sample(100, random_state=SEED), feature_perturbation="interventional",
                               model_output="probability")
sv = explainer(X)
mean_abs = pd.DataFrame({"women": np.abs(sv.values[women]).mean(axis=0), "men": np.abs(sv.values[~women]).mean(axis=0)},
                        index=FEATURES).sort_values("men")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
mean_abs.plot.barh(ax=axes[0], color=[VIOLET, INK], width=0.8)
axes[0].set_xlabel("mean |SHAP| (probability)")
axes[0].set_title("What drives predictions in each group")
flipped = X.assign(sex_male=1 - X["sex_male"])
delta = model.predict_proba(flipped)[:, 1] - model.predict_proba(X)[:, 1]
axes[1].hist(delta[women], bins=25, color=VIOLET, alpha=0.75, label="women, recorded as men")
axes[1].hist(delta[~women], bins=25, color=INK, alpha=0.6, label="men, recorded as women")
axes[1].axvline(0, color=MUTED, lw=1)
axes[1].set_xlabel("change in predicted risk when only the sex variable is flipped")
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()
crossings = ((model.predict_proba(flipped)[:, 1] >= 0.5) != (model.predict_proba(X)[:, 1] >= 0.5)).mean()
print(f"mean absolute change {np.abs(delta).mean():.3f}; decisions that change at threshold 0.5: {crossings:.1%}")

The flip probe is a simple counterfactual fairness check [13]: it asks whether the model would decide differently for the same patient recorded with the other sex. Removing the variable, "fairness through unawareness", does not settle the question, because other variables can carry the same information. The comparison below retrains the model without sex and repeats the audit.

In [ ]:
p_oof_nosex = cross_val_predict(new_model(), X.drop(columns="sex_male"), y, cv=cv, method="predict_proba")[:, 1]
pd.concat({"with sex": audit_table.loc[["AUROC", "sensitivity", "specificity"]],
           "without sex": audit(p_oof_nosex).loc[["AUROC", "sensitivity", "specificity"]]})

## 3. A model card

A model card documents what a model is for, how it was evaluated and where it must not be used [4]. The card below is generated from the numbers computed in this notebook and extended with the sections that an FDA submission or an EU technical file would expect: intended use, subgroup performance, explainability, cybersecurity and change control. It is saved as a Markdown file that can be attached to a repository or a quality management record.

In [ ]:
auc_all = roc_auc_score(y, p_oof)
top = pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES).sort_values(ascending=False).index[:5]
card = f"""# Model card: coronary artery disease risk, IEEE CARS 2026 workshop example

**Model.** Gradient boosting ensemble (150 trees of depth 3) predicting angiographic coronary artery disease
from 13 clinical variables. Version generated on {pd.Timestamp.today():%Y-%m-%d} for teaching purposes.

**Intended use.** Education and research on explainable AI. The model is **not** a medical device, has not been
clinically validated, and must not be used for decisions about patients.

**Training data.** UCI Cleveland heart disease data (1988): {len(y)} patients referred for angiography,
{women.sum()} women and {(~women).sum()} men, {y.mean():.0%} with disease. Single centre, historical, and small.

**Performance.** Out-of-fold AUROC {auc_all:.2f} (five-fold cross-validation). Subgroup results with 95 percent
bootstrap intervals:

{audit_table.to_markdown()}

**Explainability.** Global drivers by mean absolute SHAP value: {", ".join(top)}. Local explanations are
available as SHAP waterfalls; the deletion, stability and agreement checks of NB2 apply.

**Fairness.** Flipping only the sex variable changes the decision for {crossings:.1%} of patients. Removing the
variable does not remove group differences, because correlated variables remain.

**Robustness and cybersecurity.** The attack notebooks show that decision-based and score-based attackers can
change alerts with input changes inside measurement noise while validation metrics stay unchanged. A deployment
would need input integrity controls, logging of inputs and explanations, and sequential monitoring of
explanation-based signals.

**Limitations.** Historical data, referral population, few women, no external validation.

**Change control.** Any retraining would follow a predetermined change control plan: the same audit, the same
explanation checks, and a documented comparison with this card.

**Contact.** Prof. Dr. Utku Köse, github.com/utkukose
"""
(DATA_DIR / "model_card.md").write_text(card, encoding="utf-8")
display(Markdown(card))

## 4. The dashboard

Explanations only help when they reach the person who decides. The mini dashboard below shows, for a test patient, the predicted risk, the decision at an adjustable threshold, the SHAP explanation, and what the model would say if the patient's sex were recorded differently. The full Streamlit dashboard in the repository adds population views, the subgroup audit and the model card; run it locally with `pip install -r requirements.txt` followed by `streamlit run dashboard/app.py`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
dash_model = new_model().fit(X_train, y_train)
dash_explainer = shap.TreeExplainer(dash_model, data=X_train.sample(100, random_state=SEED),
                                    feature_perturbation="interventional", model_output="probability")


def dashboard(patient=0, threshold=0.5):
    x = X_test.iloc[[int(patient)]]
    p = dash_model.predict_proba(x)[0, 1]
    p_flip = dash_model.predict_proba(x.assign(sex_male=1 - x["sex_male"]))[0, 1]
    decision = "refer for angiography" if p >= threshold else "no referral suggested"
    print(f"Patient {int(patient)} · true diagnosis: {'disease' if y_test[int(patient)] else 'no disease'}")
    print(f"Predicted risk {p:.2f} at threshold {threshold:.2f}: {decision}")
    print(f"Same patient recorded with the other sex: risk {p_flip:.2f}")
    shap.plots.waterfall(dash_explainer(x)[0], max_display=9, show=False)
    plt.show()


widgets.interact(dashboard, patient=widgets.IntSlider(value=0, min=0, max=len(X_test) - 1, continuous_update=False),
                 threshold=widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, continuous_update=False));

## 5. Where to go next

The two hours of this workshop sample a larger body of material. The graduate course *Explainable AI in Healthcare* at Universidad Panamericana develops the same themes over 23 notebooks in six modules: [github.com/utkukose/xai-healthcare-NB-lecture](https://github.com/utkukose/xai-healthcare-NB-lecture). GEMEX, the geometric explainer used throughout Part 3, is available on PyPI as `gemex`. The browser tools of the [XAI Lab](https://utkukose.github.io/xai-cars2026-NB-workshop/) remain online after the session.

## Discussion

1. Which of the six obligations in the table would be hardest to meet for a hospital that builds its own model?
2. The audit intervals for women are wide. Is the right response more data, a different model, or a restriction of the intended use?
3. What should a model card say about attacks that the manufacturer has tested but cannot prevent?

## Summary

Explanations become governance when they are turned into evidence: a documented risk classification, a subgroup audit with honest uncertainty, a model card that states limits as clearly as performance, and an interface that shows clinicians why a recommendation was made. The same explanations that support those artefacts also carried the security signals of Part 3, which makes explainability a shared foundation for trust, oversight and defence.

## References

[1] Regulation (EU) 2024/1689 of the European Parliament and of the Council laying down harmonised rules on artificial intelligence (Artificial Intelligence Act), *Official Journal of the European Union*, L series, 12 July 2024.

[2] Regulation (EU) 2026/1744 (Digital Omnibus on AI), amending Regulation (EU) 2024/1689, published in the *Official Journal of the European Union* on 24 July 2026 and in force since 27 July 2026.

[3] Regulation (EU) 2017/745 of the European Parliament and of the Council on medical devices, *Official Journal of the European Union*, L 117, 5 May 2017.

[4] M. Mitchell et al., "Model cards for model reporting," in *Proc. Conf. Fairness, Accountability, and Transparency (FAT\*)*, 2019, pp. 220–229. doi: 10.1145/3287560.3287596

[5] Z. Obermeyer, B. Powers, C. Vogeli, and S. Mullainathan, "Dissecting racial bias in an algorithm used to manage the health of populations," *Science*, vol. 366, no. 6464, pp. 447–453, 2019. doi: 10.1126/science.aax2342

[6] L. Seyyed-Kalantari, H. Zhang, M. B. A. McDermott, I. Y. Chen, and M. Ghassemi, "Underdiagnosis bias of artificial intelligence algorithms applied to chest radiographs in under-served patient populations," *Nature Medicine*, vol. 27, pp. 2176–2182, 2021. doi: 10.1038/s41591-021-01595-0

[7] U.S. Food and Drug Administration, Health Canada, and Medicines and Healthcare products Regulatory Agency, "Transparency for Machine Learning-Enabled Medical Devices: Guiding Principles," June 2024.

[8] U.S. Food and Drug Administration, "Marketing Submission Recommendations for a Predetermined Change Control Plan for Artificial Intelligence-Enabled Device Software Functions," final guidance, December 2024.

[9] U.S. Food and Drug Administration, "Cybersecurity in Medical Devices: Quality Management System Considerations and Content of Premarket Submissions," final guidance, February 2026.

[10] A. Janosi, W. Steinbrunn, M. Pfisterer, and R. Detrano, "Heart Disease," UCI Machine Learning Repository, 1989. doi: 10.24432/C52P4X

[11] S. M. Lundberg and S.-I. Lee, "A unified approach to interpreting model predictions," in *Advances in Neural Information Processing Systems 30*, 2017, pp. 4765–4774. arXiv:1705.07874

[12] B. Efron and R. J. Tibshirani, *An Introduction to the Bootstrap*. New York: Chapman & Hall, 1993.

[13] M. J. Kusner, J. Loftus, C. Russell, and R. Silva, "Counterfactual fairness," in *Advances in Neural Information Processing Systems 30*, 2017. arXiv:1703.06856